# Data Cleaning and Integration

This notebook prepares the selected dissertation datasets for integration into a single analysis-ready dataset.

The datasets used are:

- NESO Historic Demand Data 2023
- Carbon Intensity API data 2023
- ONS System Average Price of Gas data 2023

The aim of this notebook is to clean the datasets, align their time frequency, create useful time-based variables, and save a final integrated dataset for exploratory analysis and forecasting.

In [1]:
# Import the libraries needed for data cleaning and integration

import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
# Set the main project folder paths

project_folder = Path.cwd().parent

raw_data_folder = project_folder / "Data" / "raw"
processed_data_folder = project_folder / "Data" / "processed"
outputs_folder = project_folder / "Outputs"
tables_folder = outputs_folder / "tables"
figures_folder = outputs_folder / "figures"

print("Project folder:")
print(project_folder)

print("\nRaw data folder:")
print(raw_data_folder)

print("\nProcessed data folder:")
print(processed_data_folder)

Project folder:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project

Raw data folder:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Data/raw

Processed data folder:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Data/processed


## 1. Load the Processed Datasets

This section loads the cleaned datasets created during the data source review stage. These datasets will be checked before they are aligned and integrated into a single daily dataset.

In [3]:
# Load the processed datasets for integration

neso_daily_file = processed_data_folder / "neso_demand_wind_solar_2023_daily.csv"
carbon_clean_file = processed_data_folder / "carbon_intensity_2023_clean.csv"
gas_clean_file = processed_data_folder / "ons_sap_gas_2023_clean.csv"

neso_daily = pd.read_csv(neso_daily_file)
carbon_clean = pd.read_csv(carbon_clean_file)
gas_clean = pd.read_csv(gas_clean_file)

print("NESO daily data:")
print(neso_daily.shape)

print("\nCarbon intensity data:")
print(carbon_clean.shape)

print("\nONS gas price data:")
print(gas_clean.shape)

NESO daily data:
(365, 4)

Carbon intensity data:
(17429, 5)

ONS gas price data:
(351, 3)


In [4]:
# Check column names and data types for the loaded datasets

print("NESO daily columns and data types:")
print(neso_daily.dtypes)

print("\nCarbon intensity columns and data types:")
print(carbon_clean.dtypes)

print("\nONS gas price columns and data types:")
print(gas_clean.dtypes)

NESO daily columns and data types:
date                 object
national_demand     float64
wind_generation     float64
solar_generation    float64
dtype: object

Carbon intensity columns and data types:
from                   object
to                     object
forecast_intensity      int64
actual_intensity      float64
intensity_index        object
dtype: object

ONS gas price columns and data types:
date                                   object
sap_actual_p_per_kwh                  float64
sap_7day_rolling_average_p_per_kwh    float64
dtype: object


In [5]:
# Convert date and datetime columns into proper datetime format

neso_daily["date"] = pd.to_datetime(neso_daily["date"])

carbon_clean["from"] = pd.to_datetime(carbon_clean["from"])
carbon_clean["to"] = pd.to_datetime(carbon_clean["to"])

gas_clean["date"] = pd.to_datetime(gas_clean["date"])

print("NESO daily date type:")
print(neso_daily["date"].dtype)

print("\nCarbon intensity from/to date types:")
print(carbon_clean[["from", "to"]].dtypes)

print("\nONS gas price date type:")
print(gas_clean["date"].dtype)

NESO daily date type:
datetime64[ns]

Carbon intensity from/to date types:
from    datetime64[ns, UTC]
to      datetime64[ns, UTC]
dtype: object

ONS gas price date type:
datetime64[ns]


In [6]:
# Create a daily date column for the carbon intensity dataset

carbon_clean["date"] = carbon_clean["from"].dt.date
carbon_clean["date"] = pd.to_datetime(carbon_clean["date"])

print("Carbon intensity columns after creating date column:")
print(carbon_clean.dtypes)

carbon_clean.head()

Carbon intensity columns after creating date column:
from                  datetime64[ns, UTC]
to                    datetime64[ns, UTC]
forecast_intensity                  int64
actual_intensity                  float64
intensity_index                    object
date                       datetime64[ns]
dtype: object


,from,to,forecast_intensity,actual_intensity,intensity_index,date
0,2023-01-01 00:00:00+00:00,2023-01-01 00:30:00+00:00,73,72.0,low,2023-01-01
1,2023-01-01 00:30:00+00:00,2023-01-01 01:00:00+00:00,63,80.0,low,2023-01-01
2,2023-01-01 01:00:00+00:00,2023-01-01 01:30:00+00:00,71,72.0,low,2023-01-01
3,2023-01-01 01:30:00+00:00,2023-01-01 02:00:00+00:00,76,65.0,low,2023-01-01
4,2023-01-01 02:00:00+00:00,2023-01-01 02:30:00+00:00,72,65.0,low,2023-01-01


In [7]:
# Create a daily carbon intensity dataset

daily_carbon = carbon_clean.groupby("date").agg({
    "forecast_intensity": "mean",
    "actual_intensity": "mean"
}).reset_index()

daily_carbon = daily_carbon.rename(columns={
    "forecast_intensity": "daily_average_forecast_carbon_intensity",
    "actual_intensity": "daily_average_actual_carbon_intensity"
})

print("Daily carbon intensity data:")
print(daily_carbon.shape)

daily_carbon.head()

Daily carbon intensity data:
(364, 3)


,date,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity
0,2023-01-01,106.583333,107.895833
1,2023-01-02,164.104167,157.062500
2,2023-01-03,100.500000,101.604167
3,2023-01-04,65.166667,65.500000
4,2023-01-05,103.833333,105.229167


In [8]:
# Identify missing dates in the daily carbon intensity dataset

expected_2023_dates = pd.date_range(
    start="2023-01-01",
    end="2023-12-31",
    freq="D"
)

carbon_dates = daily_carbon["date"]

missing_carbon_dates = expected_2023_dates.difference(carbon_dates)

print("Expected daily dates:", len(expected_2023_dates))
print("Actual daily carbon dates:", len(carbon_dates))
print("Missing daily carbon dates:", len(missing_carbon_dates))

missing_carbon_dates

Expected daily dates: 365
Actual daily carbon dates: 364
Missing daily carbon dates: 1


DatetimeIndex(['2023-10-21'], dtype='datetime64[ns]', freq='D')

The daily carbon intensity dataset contains 364 dates rather than 365. The missing date is 21 October 2023, which matches the earlier data source review where missing half-hourly API records were identified between 20 October and 22 October 2023. This missing date will be handled during the integration stage.

In [9]:
# Check missing values in the daily carbon intensity dataset

print("Missing values in daily carbon intensity data:")
print(daily_carbon.isna().sum())

daily_carbon[daily_carbon.isna().any(axis=1)]

Missing values in daily carbon intensity data:
date                                       0
daily_average_forecast_carbon_intensity    0
daily_average_actual_carbon_intensity      0
dtype: int64


,date,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity


The daily carbon intensity dataset does not contain missing values in the dates that are present. Although 21 October 2023 is missing from the Carbon Intensity API data, the remaining daily records all contain valid forecast and actual carbon intensity averages.

In [10]:
# Check NESO daily dataset coverage and missing values

print("NESO daily date range:")
print("Start date:", neso_daily["date"].min())
print("End date:", neso_daily["date"].max())

print("\nNESO daily shape:")
print(neso_daily.shape)

print("\nMissing values in NESO daily data:")
print(neso_daily.isna().sum())

neso_daily.head()

NESO daily date range:
Start date: 2023-01-01 00:00:00
End date: 2023-12-31 00:00:00

NESO daily shape:
(365, 4)

Missing values in NESO daily data:
date                0
national_demand     0
wind_generation     0
solar_generation    0
dtype: int64


,date,national_demand,wind_generation,solar_generation
0,2023-01-01,24189.979167,1733.208333,245.729167
1,2023-01-02,27005.520833,1122.416667,733.750000
2,2023-01-03,29646.312500,2927.687500,92.479167
3,2023-01-04,27967.145833,3670.312500,336.729167
4,2023-01-05,29392.500000,2528.229167,228.479167


The NESO daily dataset covers the full 2023 calendar year, from 1 January 2023 to 31 December 2023. It contains 365 daily records and no missing values in the selected demand, wind generation or solar generation variables.

In [11]:
# Check ONS gas dataset coverage and missing values

print("ONS gas date range:")
print("Start date:", gas_clean["date"].min())
print("End date:", gas_clean["date"].max())

print("\nONS gas shape:")
print(gas_clean.shape)

print("\nMissing values in ONS gas data:")
print(gas_clean.isna().sum())

gas_clean.head()

ONS gas date range:
Start date: 2023-01-01 00:00:00
End date: 2023-12-17 00:00:00

ONS gas shape:
(351, 3)

Missing values in ONS gas data:
date                                  0
sap_actual_p_per_kwh                  0
sap_7day_rolling_average_p_per_kwh    0
dtype: int64


,date,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh
0,2023-01-01,5.7764,5.9602
1,2023-01-02,5.9978,5.9704
2,2023-01-03,5.6898,5.9326
3,2023-01-04,5.0746,5.8222
4,2023-01-05,5.0837,5.7133


The ONS gas price dataset covers the period from 1 January 2023 to 17 December 2023. It contains 351 daily records and no missing values in the selected gas price variables. The dataset does not cover the final two weeks of December 2023, so the integrated modelling dataset will need to account for this difference in date coverage.

In [12]:
# Merge the daily datasets into one integrated dataset

integrated_data = neso_daily.merge(
    daily_carbon,
    on="date",
    how="inner"
)

integrated_data = integrated_data.merge(
    gas_clean,
    on="date",
    how="inner"
)

print("Integrated dataset shape:")
print(integrated_data.shape)

print("\nIntegrated dataset date range:")
print("Start date:", integrated_data["date"].min())
print("End date:", integrated_data["date"].max())

integrated_data.head()

Integrated dataset shape:
(350, 8)

Integrated dataset date range:
Start date: 2023-01-01 00:00:00
End date: 2023-12-17 00:00:00


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh
0,2023-01-01,24189.979167,1733.208333,245.729167,106.583333,107.895833,5.7764,5.9602
1,2023-01-02,27005.520833,1122.416667,733.750000,164.104167,157.062500,5.9978,5.9704
2,2023-01-03,29646.312500,2927.687500,92.479167,100.500000,101.604167,5.6898,5.9326
3,2023-01-04,27967.145833,3670.312500,336.729167,65.166667,65.500000,5.0746,5.8222
4,2023-01-05,29392.500000,2528.229167,228.479167,103.833333,105.229167,5.0837,5.7133


The three daily datasets were merged using an inner join on the date column. This means that only dates available in all three datasets were retained. The final integrated dataset contains 350 daily records from 1 January 2023 to 17 December 2023. The date range is limited by the ONS gas price dataset, which ends on 17 December 2023, and the missing Carbon Intensity API date on 21 October 2023 is also excluded from the integrated dataset.

In [13]:
# Check missing values in the integrated dataset

print("Missing values in integrated dataset:")
print(integrated_data.isna().sum())

integrated_data[integrated_data.isna().any(axis=1)]

Missing values in integrated dataset:
date                                       0
national_demand                            0
wind_generation                            0
solar_generation                           0
daily_average_forecast_carbon_intensity    0
daily_average_actual_carbon_intensity      0
sap_actual_p_per_kwh                       0
sap_7day_rolling_average_p_per_kwh         0
dtype: int64


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh


The integrated dataset does not contain any missing values after merging. This confirms that the inner join successfully retained only the dates where NESO demand and renewable generation data, Carbon Intensity API data and ONS gas price data are all available.

In [14]:
# Create time-based features for analysis and modelling

integrated_data["day"] = integrated_data["date"].dt.day
integrated_data["month"] = integrated_data["date"].dt.month
integrated_data["weekday"] = integrated_data["date"].dt.day_name()
integrated_data["weekday_number"] = integrated_data["date"].dt.weekday
integrated_data["is_weekend"] = integrated_data["weekday_number"].isin([5, 6])

print("Integrated dataset shape after adding time-based features:")
print(integrated_data.shape)

integrated_data.head()

Integrated dataset shape after adding time-based features:
(350, 13)


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh,day,month,weekday,weekday_number,is_weekend
0,2023-01-01,24189.979167,1733.208333,245.729167,106.583333,107.895833,5.7764,5.9602,1,1,Sunday,6,True
1,2023-01-02,27005.520833,1122.416667,733.750000,164.104167,157.062500,5.9978,5.9704,2,1,Monday,0,False
2,2023-01-03,29646.312500,2927.687500,92.479167,100.500000,101.604167,5.6898,5.9326,3,1,Tuesday,1,False
3,2023-01-04,27967.145833,3670.312500,336.729167,65.166667,65.500000,5.0746,5.8222,4,1,Wednesday,2,False
4,2023-01-05,29392.500000,2528.229167,228.479167,103.833333,105.229167,5.0837,5.7133,5,1,Thursday,3,False


Additional time-based features were created from the date column. These include the day of the month, month number, weekday name, weekday number and a weekend indicator. These variables will support later exploratory analysis and modelling because electricity demand, renewable generation, carbon intensity and gas price conditions may vary across weekdays, weekends and seasons.

In [15]:
# Check the final columns in the integrated dataset

print("Final integrated dataset columns:")
print(integrated_data.columns.tolist())

Final integrated dataset columns:
['date', 'national_demand', 'wind_generation', 'solar_generation', 'daily_average_forecast_carbon_intensity', 'daily_average_actual_carbon_intensity', 'sap_actual_p_per_kwh', 'sap_7day_rolling_average_p_per_kwh', 'day', 'month', 'weekday', 'weekday_number', 'is_weekend']


In [16]:
# Save the integrated dataset for later analysis and modelling

integrated_data_file = processed_data_folder / "integrated_energy_market_risk_2023.csv"

integrated_data.to_csv(integrated_data_file, index=False)

print("Saved integrated dataset to:")
print(integrated_data_file)

Saved integrated dataset to:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Data/processed/integrated_energy_market_risk_2023.csv


In [17]:
# Read back the saved integrated dataset to confirm it saved correctly

saved_integrated_data = pd.read_csv(integrated_data_file)

print("Saved integrated dataset shape:")
print(saved_integrated_data.shape)

saved_integrated_data.head()

Saved integrated dataset shape:
(350, 13)


,date,national_demand,wind_generation,solar_generation,daily_average_forecast_carbon_intensity,daily_average_actual_carbon_intensity,sap_actual_p_per_kwh,sap_7day_rolling_average_p_per_kwh,day,month,weekday,weekday_number,is_weekend
0,2023-01-01,24189.979167,1733.208333,245.729167,106.583333,107.895833,5.7764,5.9602,1,1,Sunday,6,True
1,2023-01-02,27005.520833,1122.416667,733.750000,164.104167,157.062500,5.9978,5.9704,2,1,Monday,0,False
2,2023-01-03,29646.312500,2927.687500,92.479167,100.500000,101.604167,5.6898,5.9326,3,1,Tuesday,1,False
3,2023-01-04,27967.145833,3670.312500,336.729167,65.166667,65.500000,5.0746,5.8222,4,1,Wednesday,2,False
4,2023-01-05,29392.500000,2528.229167,228.479167,103.833333,105.229167,5.0837,5.7133,5,1,Thursday,3,False


The saved integrated dataset was read back into the notebook to confirm that it was exported successfully. The saved file contains 350 rows and 13 columns, matching the expected shape of the final integrated dataset.

In [18]:
# Create a summary table for the final integrated dataset

integration_summary = pd.DataFrame([{
    "dataset": "Integrated energy market risk dataset",
    "file_name": "integrated_energy_market_risk_2023.csv",
    "rows": integrated_data.shape[0],
    "columns": integrated_data.shape[1],
    "start_date": integrated_data["date"].min(),
    "end_date": integrated_data["date"].max(),
    "missing_values": integrated_data.isna().sum().sum()
}])

integration_summary

,dataset,file_name,rows,columns,start_date,end_date,missing_values
0,Integrated energy market risk dataset,integrated_energy_market_risk_2023.csv,350,13,2023-01-01,2023-12-17,0


In [19]:
# Save the integration summary table

integration_summary_file = tables_folder / "integration_summary.csv"

integration_summary.to_csv(integration_summary_file, index=False)

print("Saved integration summary table to:")
print(integration_summary_file)


Saved integration summary table to:
/Users/tosinjimoh/Downloads/Msc/Lecture Notes/2026-MSc-Dissertation-Project/Outputs/tables/integration_summary.csv


The integration summary table was saved to the Outputs/tables folder. This provides a concise record of the final integrated dataset, including the file name, number of rows, number of columns, date range and missing value count.

## 2. Notebook Summary

This notebook prepared the processed dissertation datasets for integration. The NESO daily demand, wind generation and solar generation dataset was loaded alongside the cleaned Carbon Intensity API dataset and the cleaned ONS gas price dataset.

The Carbon Intensity API data was aggregated from half-hourly observations to daily averages so that it could be aligned with the daily NESO and ONS gas datasets. The datasets were then merged using the date column and an inner join, retaining only dates available across all three sources.

The final integrated dataset contains 350 daily records and 13 columns, covering the period from 1 January 2023 to 17 December 2023. The final date range is limited by the ONS gas price dataset, and the missing Carbon Intensity API date on 21 October 2023 is excluded from the merged dataset.

The integrated dataset contains no missing values and has been saved as `integrated_energy_market_risk_2023.csv` in the processed data folder. This dataset will be used in the next stage for exploratory data analysis and later modelling.